In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


class NN:

    def __init__(self, sizes):
        self.layers = len(sizes)
        self.weights = [
            np.random.randn(row, col) for row, col in zip(sizes[1:], sizes[:-1])
        ]
        self.biases = [np.random.randn(row, 1) for row in sizes[1:]]

    def feedforward(self, X):
        aL = X.T
        for weight, bias in zip(self.weights, self.biases):
            aL = sigmoid(weight @ aL + bias)
        return aL

    def backprop(self, X, Y, eta=0.01):
        N = len(X)
        L = self.layers
        activations = [X.T]

        for weight, bias in zip(self.weights, self.biases):
            activations.append(sigmoid(weight @ activations[-1] + bias))

        del_l = activations[-1] - Y.T

        for layer in range(1, L):

            wL = self.weights[-layer]
            bL = self.biases[-layer]

            del_C_w = del_l @ activations[-layer - 1].T
            del_C_b = np.sum(del_l, axis=1, keepdims=True)
            del_l = (wL.T @ del_l) * (
                activations[-layer - 1] * (1 - activations[-layer - 1])
            )

            wL -= (eta / N) * del_C_w
            bL -= (eta / N) * del_C_b

    def predict(self, X):
        output = self.feedforward(X)
        return (output >= 0.5).astype(int).flatten()

    def train(self, X_train, Y_train, epochs=1000, eta=0.5):
        errors = []
        for _ in range(epochs):
            self.backprop(X_train, Y_train, eta=eta)
            predictions = self.feedforward(X_train).flatten()
            loss = -np.mean(
                Y_train * np.log(predictions) + (1 - Y_train) * np.log(1 - predictions)
            )
            errors.append(loss)

        plt.figure(figsize=(8, 5))
        plt.plot(range(1, epochs + 1), errors)
        plt.xlabel("Epoch")
        plt.ylabel("Binary Cross-Entropy Loss")
        plt.title("Neural Network Training Error")
        plt.grid()
        plt.show()



In [ ]:
df = pd.read_csv("mushrooms.csv")
Y = (df["class"] == "p").astype(int).to_numpy()

X_df = df.drop(columns=["class"])
X_df = pd.get_dummies(X_df, dtype=int)

X = X_df.to_numpy()


np.random.seed(42)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, stratify=Y
)

net = NN([X.shape[1], 3, 2, 1])

net.train(X, Y, epochs=1000, eta=0.5)
nn_predictions = net.predict(X_test)
nn_accuracy = accuracy_score(Y_test, nn_predictions)
nn_error = 1 - nn_accuracy

print(f"Accuracy : {nn_accuracy * 100:.2f}%")
print(f"Error    : {nn_error * 100:.2f}%")